# Analyse de la pauvreté des ménages en RDC à partir de l'ECVM 2024

## Génération des dictionnaires de variables

Dans cette étape, nous construisons automatiquement des dictionnaires de variables à partir des bases de données ECVM (individu, ménage et welfare).

L’objectif est de documenter la structure des données afin de faciliter leur compréhension, leur préparation et leur exploitation dans les analyses ultérieures.

Le processus repose sur les étapes suivantes :

- Chargement des bases de données au format Stata (.dta)
- Extraction des noms de variables
- Récupération des descriptions associées aux variables (labels Stata)
- Identification du type de variable (qualitative ou quantitative)
- Extraction des modalités pour les variables qualitatives
- Construction d’un tableau structuré regroupant :
  - le nom de la variable
  - sa description
  - son type
  - les codes des modalités
  - les libellés associés

Les dictionnaires sont ensuite exportés au format Excel afin de permettre une consultation externe et une meilleure traçabilité des données utilisées.

Cette étape constitue une phase essentielle du prétraitement, garantissant une utilisation cohérente et documentée des variables dans les analyses statistiques et les modèles de machine learning.

In [15]:
# Chargement des bibliothèques nécessaires
import pandas as pd
import numpy as np
import os


In [66]:
# Dossier
folder = r"C:\Users\LENOVO\Desktop\memoire\data"

# Liste des fichiers
files = {
    "individu": "ecvm_individu_rdc_2024.dta",
    "menage": "ecvm_menage_rdc_2024.dta",
    "welfare": "ecvm_welfare_rdc_2024.dta"
}

# --- 1. STOCKAGE DES BASES ---
bases = {}            # DataFrames
dictionnaires = {}    # Dictionnaires variables/modalités

for nom, file in files.items():
    path = os.path.join(folder, file)

    # Charger la base
    df = pd.read_stata(path, convert_categoricals=True)
    bases[nom] = df

    # Charger les labels variables
    with pd.read_stata(path, iterator=True) as reader:
        var_labels = reader.variable_labels()

    # --- Dictionnaire des variables : toutes les variables ---
    df_var = pd.DataFrame({
        "Variable": df.columns
    })

    # Ajouter les descriptions si elles existent
    df_var["Description_variable"] = df_var["Variable"].map(var_labels)

    # --- Modalités ---
    rows = []

    for col in df.columns:
        if str(df[col].dtype) == "category":
            for code, label in enumerate(df[col].cat.categories):
                rows.append([col, code, str(label)])

    df_val = pd.DataFrame(rows, columns=["Variable", "Code", "Label_modalite"])

    # --- Fusion : on garde toutes les variables, même sans modalités ---
    df_final = df_var.merge(df_val, on="Variable", how="left")

    # --- Type de variable ---
    df_final["Type_variable"] = df_final["Variable"].apply(
        lambda x: "Qualitative" if x in df_val["Variable"].unique() else "Quantitative / autre"
    )

    # --- Nettoyage ---
    df_final["Description_variable"] = df_final["Description_variable"].fillna("Non renseigné")
    df_final = df_final.sort_values(["Variable", "Code"], na_position="last").reset_index(drop=True)

    # Réorganiser les colonnes
    df_final = df_final[
        ["Variable", "Description_variable", "Type_variable", "Code", "Label_modalite"]
    ]

    # Stocker dictionnaire
    dictionnaires[nom] = df_final

    # Export Excel
    output = os.path.join(folder, f"dictionnaire_{nom}.xlsx")
    df_final.to_excel(output, index=False)

# --- 2. ACCÈS FACILE AUX BASES ---
df_individu = bases["individu"]
df_menage = bases["menage"]
df_welfare = bases["welfare"]

# --- 3. ACCÈS AUX DICTIONNAIRES ---
dico_individu = dictionnaires["individu"]
dico_menage = dictionnaires["menage"]
dico_welfare = dictionnaires["welfare"]

# --- 4. TEST ---
print(df_individu.head())
print(dico_welfare.head(20))

  country  year    hhid  grappe  menage  numind    regmil    region  province  \
0     rdc  2024  1001.0       1       1       1  Kinshasa  Kinshasa  Kinshasa   
1     rdc  2024  1001.0       1       1       2  Kinshasa  Kinshasa  Kinshasa   
2     rdc  2024  1001.0       1       1       3  Kinshasa  Kinshasa  Kinshasa   
3     rdc  2024  1001.0       1       1       4  Kinshasa  Kinshasa  Kinshasa   
4     rdc  2024  1001.0       1       1       5  Kinshasa  Kinshasa  Kinshasa   

         commune  ... sectins_sec  csp_sec volhor_sec salaire_sec  bank  \
0  Bandalungwa_1  ...         NaN      NaN        NaN         NaN   Oui   
1  Bandalungwa_1  ...         NaN      NaN        NaN         NaN   Oui   
2  Bandalungwa_1  ...         NaN      NaN        NaN         NaN   Non   
3  Bandalungwa_1  ...         NaN      NaN        NaN         NaN   Non   
4  Bandalungwa_1  ...         NaN      NaN        NaN         NaN   Non   

        serviceconsult          persconsult       nivie quinte

In [67]:
# La taille des bases
print(df_individu.shape)
print(df_menage.shape)
print(df_welfare.shape)

(120476, 56)
(24522, 43)
(24521, 42)
